# 🎙️ LuyenAm - AI Voice Conversion (Seed-VC trên Google Colab GPU)

Notebook này dùng **Seed-VC** (Zero-shot Voice Conversion) để chuyển đổi các câu đọc mẫu bản xứ trong `sample_audio/` sang **chất giọng thật của bạn** (giữ nguyên phát âm chuẩn, ngữ điệu tự nhiên và điều chỉnh cao độ `auto-f0-adjust` theo giọng bạn).

**Trước khi chạy:** Chọn menu **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# @title 1. Kiểm tra GPU & Cài đặt Seed-VC + Lấy kho câu mẫu từ GitHub
!nvidia-smi
!git clone --depth 1 https://github.com/Plachtaa/seed-vc.git /content/seed-vc
!git clone --depth 1 https://github.com/tranductinh2001/LuyenAm.git /content/LuyenAm
%cd /content/seed-vc
!pip install -q munch einops descript-audio-codec pydub resemblyzer hydra-core huggingface-hub soundfile pyyaml accelerate
print('✅ Cài đặt hoàn tất!')

In [ ]:
# @title 2. Tải lên file giọng mẫu của bạn (My_Voice_Sample_Colab.wav)
from google.colab import files
import os, shutil

os.makedirs('/content/input_voice', exist_ok=True)
uploaded = files.upload()
ref_voice_path = None
for fname in uploaded.keys():
    dst = os.path.join('/content/input_voice', fname)
    shutil.move(fname, dst)
    ref_voice_path = dst
    print(f'✅ Đã nhận giọng mẫu: {ref_voice_path}')

In [ ]:
# @title 3. Chạy AI Chuyển Đổi Giọng (Seed-VC) cho Bài Audio đã chọn
AUDIO_NUMBER = "1" # @param ["1", "2", "3", "4", "5", "all"]
DIFFUSION_STEPS = 30 # @param {type:"slider", min:15, max:50, step:5}
CFG_RATE = 0.7 # @param {type:"slider", min:0.3, max:1.0, step:0.1}

import glob, os, subprocess, shutil

assert ref_voice_path and os.path.exists(ref_voice_path), '❌ Hãy chạy bước 2 để tải lên My_Voice_Sample_Colab.wav trước!'

sample_dir = '/content/LuyenAm/sample_audio'
out_dir = '/content/converted_output'
os.makedirs(out_dir, exist_ok=True)

pattern = 'audio_*.mp3' if AUDIO_NUMBER == 'all' else f'audio_{AUDIO_NUMBER}_*.mp3'
src_files = sorted(glob.glob(os.path.join(sample_dir, pattern)))
print(f'🔍 Tìm thấy {len(src_files)} câu mẫu cần chuyển giọng...')

%cd /content/seed-vc
for idx, src in enumerate(src_files, 1):
    base = os.path.splitext(os.path.basename(src))[0]
    tmp_out = f'/content/tmp_vc_{base}'
    tmp_src = f'/content/tmp_src_{base}.wav'
    os.makedirs(tmp_out, exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-i', src, '-ar', '22050', '-ac', '1', tmp_src], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    cmd = [
        'python', 'inference.py',
        '--source', tmp_src,
        '--target', ref_voice_path,
        '--output', tmp_out,
        '--diffusion-steps', str(DIFFUSION_STEPS),
        '--length-adjust', '1.0',
        '--inference-cfg-rate', str(CFG_RATE),
        '--f0-condition', 'False',
        '--auto-f0-adjust', 'True',
        '--fp16', 'True'
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print('❌ Lỗi inference:', res.stdout, res.stderr)
        raise RuntimeError(f'inference.py failed on {base}')
    wavs = glob.glob(os.path.join(tmp_out, '*.wav'))
    if wavs:
        final_wav = os.path.join(out_dir, f'{base}.wav')
        shutil.move(wavs[0], final_wav)
        print(f'[{idx}/{len(src_files)}] ✅ {base}.wav')
    shutil.rmtree(tmp_out, ignore_errors=True)
    if os.path.exists(tmp_src): os.remove(tmp_src)

print('🎉 Hoàn tất chuyển giọng! Chạy bước 4 để tải file ZIP về nạp vào Studio.')

In [ ]:
# @title 4. Nén & Tải về file ZIP để nạp vào LuyenAm Studio
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/converted_audio_for_studio', 'zip', '/content/converted_output')
print(f'📦 Đã đóng gói: {zip_path}')
files.download(zip_path)